In [10]:
# Main environment setup
# Comment: install only what you need in your environment.
# The original notebook used CLIP-specific installs; this notebook switches to LLaVA/TRL/PEFT.

!pip install -U datasets huggingface_hub torch torchvision Pillow numpy pandas scipy scikit-image tqdm matplotlib
!pip install -U transformers accelerate trl peft bitsandbytes tensorboard sentencepiece
# Optional:
!pip install -U hf_xet

import os
import io
import re
import ast
import json
import math
import time
import copy
import random
import pathlib
import functools
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import io
import base64

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset, DatasetDict, load_dataset
from scipy.spatial import KDTree
from skimage.draw import line_aa
from skimage.draw import line as sk_line

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration
)
from peft import (
    LoraConfig,
    IA3Config,
    get_peft_model,
    prepare_model_for_kbit_training,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"torch        : {torch.__version__}")
print(f"device       : {DEVICE}")
print(f"bf16 support : {torch.cuda.is_available() and torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")


  Using cached huggingface_hub-1.8.0-py3-none-any.whl (625 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.2
    Uninstalling huggingface-hub-0.36.2:
      Successfully uninstalled huggingface-hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.8.0 which is incompatible.
You should consider upgrading via the '/home/jairoc/llm_course/vln-project/.venv/bin/python3 -m pip install --upgrade pip' command.
  Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.8.0
    Uninstalling huggingface-hub-1.8.0:
      Successfully uninstalled huggingface-hub-1.8.0
You should consider upgrading via the '/home/jairoc/llm_course/vln-pro

In [70]:
# Global configuration

CFG = {
    # Data / task settings
    "embodiment": "Legged Robot",
    "val_split": 0.20,
    "seed": 42,

    # Quick experiment testing:
    "max_train_samples": None,      
    "max_val_samples": None,
    "max_test_samples": None,

    # Model settings:
    "model_id": "llava-hf/llava-v1.6-mistral-7b-hf",
    "image_max_side": 672,           
    "max_new_tokens": 800,
    "generation_num_beams": 1,

    # Training settings
    "num_train_epochs": 100,
    "learning_rate": 2e-5,
    "weight_decay": 1e-4,
    "warmup_ratio": 0.03,
    "patience": 10, # Stopping criteria if val loss does not improve after n epochs
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "save_total_limit": 1,
    "gradient_checkpointing": True,
    "report_to": "tensorboard",

    # Formatting / output settings
    "trace_points": None,            
    "output_root": "./llava_navitrace_runs",
    "cache_root": "./llava_navitrace_cache",
    "penalty_tsv": "./category_penalty.tsv",
    "m2f_config": "./mask2former_config.json",
    "bad_score_threshold": 3234.75,
    "penalty_dist_thr": 35,

    # Method-specific default ranks
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
}

os.makedirs(CFG["output_root"], exist_ok=True)
os.makedirs(CFG["cache_root"], exist_ok=True)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

print(CFG)


{'embodiment': 'Legged Robot', 'val_split': 0.2, 'seed': 42, 'max_train_samples': None, 'max_val_samples': None, 'max_test_samples': None, 'model_id': 'llava-hf/llava-v1.6-mistral-7b-hf', 'image_max_side': 672, 'max_new_tokens': 800, 'generation_num_beams': 1, 'num_train_epochs': 100, 'learning_rate': 2e-05, 'weight_decay': 0.0001, 'warmup_ratio': 0.03, 'patience': 10, 'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 1, 'gradient_accumulation_steps': 8, 'logging_steps': 10, 'save_strategy': 'epoch', 'eval_strategy': 'epoch', 'save_total_limit': 1, 'gradient_checkpointing': True, 'report_to': 'tensorboard', 'trace_points': None, 'output_root': './llava_navitrace_runs', 'cache_root': './llava_navitrace_cache', 'penalty_tsv': './category_penalty.tsv', 'm2f_config': './mask2former_config.json', 'bad_score_threshold': 3234.75, 'penalty_dist_thr': 35, 'lora_r': 16, 'lora_alpha': 16, 'lora_dropout': 0.05}


In [5]:
# Load NaviTrace and reproduce the original train/val filtering logic
# Main step comment: keep only samples that have at least one GT trace for the chosen embodiment.

ALL_SAMPLES_PKL = os.path.join(CFG["cache_root"], "samples_all.pkl")

if os.path.exists(ALL_SAMPLES_PKL):
    print("Loading cached filtered samples...")
    with open(ALL_SAMPLES_PKL, "rb") as f:
        saved = torch.load(f, weights_only=False) if ALL_SAMPLES_PKL.endswith(".pt") else None

if os.path.exists(ALL_SAMPLES_PKL) and saved is None:
    # Fall back to pickle if the cache was created as a pickle file in a prior run
    import pickle
    with open(ALL_SAMPLES_PKL, "rb") as f:
        saved = pickle.load(f)

if os.path.exists(ALL_SAMPLES_PKL):
    val_split_samples = saved["val_split"]
    print(f"NaviTrace filtered validation split : {len(val_split_samples)} samples")
else:
    print("Loading NaviTrace from Hugging Face...")
    dataset = load_dataset("leggedrobotics/navitrace")
    print(f"Available splits: {list(dataset.keys())}")

    val_split_samples = []
    skipped = 0
    for s in tqdm(list(dataset["validation"]), desc="Filtering validation"):
        gt = s["ground_truth"].get(CFG["embodiment"])
        if gt is not None and len(gt) > 0:
            val_split_samples.append(s)
        else:
            skipped += 1

    print(f"Kept={len(val_split_samples)}  Skipped={skipped}")

    import pickle
    with open(ALL_SAMPLES_PKL, "wb") as f:
        pickle.dump({"val_split": val_split_samples}, f)

# Compute a common number of waypoints from the median GT trace length
trace_lengths = []
for s in val_split_samples:
    for t in s["ground_truth"][CFG["embodiment"]]:
        trace_lengths.append(len(t))

CFG["trace_points"] = int(np.median(trace_lengths))
print(f"Common trace length N = {CFG['trace_points']}")

# Fixed split for reproducibility
random.seed(CFG["seed"])
random.shuffle(val_split_samples)

n_our_val = int(len(val_split_samples) * CFG["val_split"])
val_samples = val_split_samples[:n_our_val]
train_samples = val_split_samples[n_our_val:]

if CFG["max_train_samples"] is not None:
    train_samples = train_samples[:CFG["max_train_samples"]]
if CFG["max_val_samples"] is not None:
    val_samples = val_samples[:CFG["max_val_samples"]]

print(f"Train samples: {len(train_samples)}")
print(f"Val samples  : {len(val_samples)}")


Loading cached filtered samples...
NaviTrace filtered validation split : 501 samples
Common trace length N = 9
Train samples: 401
Val samples  : 100


In [6]:
# Load the test split for later leaderboard-style export
# Main step comment: keep only test samples that support the requested embodiment.

dataset = load_dataset("leggedrobotics/navitrace")

test_samples = []
for s in tqdm(list(dataset["test"]), desc="Filtering test"):
    if CFG["embodiment"] in s.get("embodiments", []):
        test_samples.append(s)

if CFG["max_test_samples"] is not None:
    test_samples = test_samples[:CFG["max_test_samples"]]

print(f"Test samples: {len(test_samples)}")


Filtering test:   0%|          | 0/500 [00:00<?, ?it/s]

Test samples: 500


### Loading in LLaVA

In [11]:
processor = AutoProcessor.from_pretrained(CFG["model_id"])

model = LlavaNextForConditionalGeneration.from_pretrained(
    CFG["model_id"],
    torch_dtype=DTYPE,
    device_map="auto"
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

/home/jairoc/llm_course/vln-project/.venv/lib64/python3.9/site-packages/accelerate/utils/modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['model.image_newline']
  warnings.warn(


### LLaVA Prompting Functions

In [32]:
def pad_to_square(image):
    w, h = image.size
    max_side = max(w, h)
    padded = Image.new("RGB", (max_side, max_side), (0, 0, 0))
    padded.paste(image, ((max_side - w) // 2, (max_side - h) // 2))
    return padded

def pred_padded_to_pixel(pred_normed, orig_w, orig_h):
    max_side = max(orig_w, orig_h)
    x_offset = (max_side - orig_w) / 2
    y_offset = (max_side - orig_h) / 2
    p = np.array(pred_normed)
    p[:, 0] = np.clip(p[:, 0] * max_side - x_offset, 0, orig_w - 1)
    p[:, 1] = np.clip(p[:, 1] * max_side - y_offset, 0, orig_h - 1)

    return p.tolist()

### Zero-shot

In [78]:
ZERO_SHOT_PROMPT = (
    "You are a visual navigation model for the NaviTrace benchmark. " 
    "Given a scene image and a navigation instruction, predict a feasible 2D path "
    "for a Legged robot as normalized waypoints in the padded-square image frame. "
    "Return JSON only with keys 'goal' and 'trace'. "
    "'goal' is the final destination [x, y]. "
    "'trace' is an ordered list of [x, y] pairs, adll values in [0, 1]. "
    "Do not add any explaintion"
)

def zero_shot_prompt(image, trace_points):
    orig_image = sample["image"].convert("RGB")
    orig_w, orig_h = orig_image.size
    padded_image = pad_to_square(orig_image)
    task = sample["task"]

    user_text = (
        f"Task: {task}\n"
        f"Embodiment: Legged Robot\n"
        f"Output exactly {trace_points} waypoints.\n"
        f"Predict the path as JSON only. "
    )

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": ZERO_SHOT_PROMPT + "\n\n" + user_text},
                {"type": "image"}
            ]
        }
    ]

    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=padded_image, text=prompt, return_tensors="pt").to(DEVICE)

    output = model.generate(**inputs, max_new_tokens=CFG["max_new_tokens"], num_beams=CFG["generation_num_beams"])
    raw = processor.decode(output[0], skip_special_tokens=True)
    # print(raw[-500:])  # Print the last 500 chars of the raw output for debugging
    return parse_trace_output(raw, orig_w, orig_h)

In [79]:
def parse_trace_output(raw, orig_w, orig_h):
    matches = list(re.finditer(r'\{[^{}]*"trace"[^{}]*\}', raw, re.DOTALL))
    
    if not matches:
        matches = list(re.finditer(r'\{.*?\}', raw, re.DOTALL))
    
    if not matches:
        print("No JSON found in model output!")
        return None
    
    
    for match in reversed(matches):
        try:
            data = json.loads(match.group())
            goal = data.get("goal")
            trace = data.get("trace")

            if goal is None or trace is None:
                continue
            trace_pixel = pred_padded_to_pixel(trace, orig_w, orig_h)
            return {"goal": goal, "trace": trace_pixel}
        except json.JSONDecodeError:
            print("Error decoding JSON from model output!")
            return None

In [ ]:
# testing the zero-shot prompt on a few val samples before training any model
for sample in val_samples:
    print(f"\nTask: {sample['task']}")

    zs_result = zero_shot_prompt(sample, CFG["trace_points"])
    print(f"Zero-shot result: {zs_result}")
    break
    

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Task: enter the backyard behind the brick wall
Zero-shot result: {'goal': [0.384, 0.667], 'trace': [[148.91000000000003, 910.455], [353.65999999999997, 910.455], [522.92, 910.455], [678.53, 910.455], [831.41, 910.455], [984.29, 910.455], [1023.0, 910.455], [353.65999999999997, 910.455]]}


### Save JSON file

In [82]:
for sample in val_samples[:5]:
    zs_result = zero_shot_prompt(sample, CFG["trace_points"])

    if zs_result is not None:
        sample_id = sample.get("sample_id", sample.get("id", "unknown"))

        output = {
            "sample_id": sample_id,
            "task": sample["task"],
            "goal": zs_result["goal"],
            "trace": zs_result["trace"],
        }

        with open(os.path.join(CFG["output_root"], f"{sample_id}_zs_output.json"), "w") as f:
            json.dump(output, f, indent=2)

            print(f"Saved zero-shot output for sample {sample_id}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saved zero-shot output for sample 7cd1e880-81e4-4407-a8df-0ebe2f16e958


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saved zero-shot output for sample 7feff4cb-45f4-47cb-8781-413bbe90e2bc


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saved zero-shot output for sample 996adb54-09e5-4f74-abe2-2a821be73ad4


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saved zero-shot output for sample 37d45e1d-5ee6-4c94-96d4-12faf1dccd80
Saved zero-shot output for sample 00c50fd8-99c3-41b9-bead-1c69a5949099
